In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import argparse
import os
import random
import argparse
from karateclub import NetLSD, Graph2Vec, FeatherGraph
from grakel import GraphKernel
import toper

from sklearn.manifold import MDS
from sklearn.decomposition import PCA

from magnipy.magnitude.distances import get_dist

from diversity_curves.process import get_mag_graphs, get_funs_df, get_mag_funs_graphs
from diversity_curves.simulate import simulate_graphs_kwargs
from diversity_curves.classifiers import knn_classifier
from diversity_curves.clustering_performance import compute_clustering_scores_resample, compute_silhouette_score_resamples, compute_rand_agglo, compute_rand_kmeans, all_clustering_results_euclidean
from diversity_curves.baselines.spectral_zoo import est_moment
from diversity_curves.baselines.ph import obtain_input_from_graphs

# scratch space creating script

In [2]:

# argparser = argparse.ArgumentParser()
# argparser.add_argument("--output_dir", type=str, default="./simulations/results/sim_params/", help="Output directory for results")
# argparser.add_argument('--models', type=str, nargs='*', default=None, help='select the models to be computed, has to be a subset of the models specifies in models_configs')
# argparser.add_argument('--ngraphs', type=int, default=3, help='Number of graphs that are generated per cardinality')
# argparser.add_argument('--nrange', type=int, nargs=2, default=[10,30], help='specify the range of cardinalities of the generated graphs')
# argparser.add_argument('--nsteps', type=int, default=1, help='Specify the stepsize for the cardinalities of the generated graphs')
# argparser.add_argument('--nseed', type=int, default=10, help='specify the number of different seeds for the ablation study')

# args = argparser.parse_args()
# results_dir = args.output_dir
n_low = 10
n_high = 30
n_steps = 1
n_graphs = 3
n_seeds = 10

# setting the parameters as reported in the paper
models_configs = {
'ER': {'ER_probs': [0.1, 0.5, 0.9], 
    'ER_nps': None}, 
'SBM': {
    'SBM_probs': [0.6, 0.01], 
    'SBM_blocks': [[1/2], [1/3, 1/3], [1/4, 1/4, 1/4]] 
    },
'RG': {
    'RG_probs': [0.25, 0.5, 0.75]
},
'RPG':{
    'RPG_probs': [[0.9, 0.01], [0.9, 0.1], [0.9, 0.5]],
    'RPG_props': [1/3, 1/3] 
},
}

models = list(models_configs.keys())

# if args.models is None:
#     models = models = list(models_configs.keys())
# else:
#     models = args.models

# choice of graph metric we use for the spread computation
graph_metrics = ['shortest_path_distance']

scores = ['acc_mean', 'silh_mean', 'ari_agglo_mean', 'db_mean',	'ch_mean', 'ari_kmeans_mean']
scores_asc = {'acc_mean': False, 'silh_mean': False, 'ari_agglo_mean': False, 'db_mean': True,'ch_mean': False, 'ari_kmeans_mean': False}

# results_dir = f'{output_dir}/nodes_{n_low}_to_{n_high}_stepsize_{n_steps}/n_graphs_{n_graphs}/'

# if not os.path.exists(results_dir):
#     os.makedirs(results_dir)


all_configs = {}
all_results = {}


In [3]:

model = models[0]

model_results = {}
scores_dfs = []
model_params = models_configs[model].keys()
config = [models_configs[model][para] for para in model_params]
model_kwargs = models_configs[model]

# setting parameters and folder for plots
sns.set_style('white')
matplotlib.rcParams.update({'font.size': 18})
# if not os.path.exists(results_dir + f'{model}/'):
#     os.makedirs(results_dir + f'{model}/')

# generate graphs
G_nx, Gs, y, y_probs, n_nodes = simulate_graphs_kwargs(
        model = model, n_graphs=n_graphs,
        nodes_range=range(n_low,n_high, n_steps),
        **model_kwargs,
        )

# labels for PCA plots
match model:
    case 'ER':
        hue_names = {str(i): f'$p={p}$' for i, p in enumerate(model_kwargs['ER_probs'])}
    case 'RG':
        hue_names = {str(i): f'$r={r}$' for i, r in enumerate(model_kwargs['RG_probs'])}
    case 'SBM':
        hue_names = {
            str(0): '$2$ communities',
            str(1): '$3$ communities',
            str(2): '$4$ communities'}
    case 'RPG':
        hue_names = {str(i): f'$p_{{in}}={p[0]}, p_{{out}}={p[1]}$' for i, p in enumerate(model_kwargs['RPG_probs'])}

compute_at_cardinality = None
upsample_to = max(n_nodes) 
downsample_to = None


In [4]:

##################################
### computing diversity curves ###
##################################

for metric in graph_metrics:

    acc_per_seed = {}
    acc_per_seed[f'divcurves_{metric}'] = []
    acc_per_seed[f'divcurves_pca_{metric}'] = []

    mag_funs_per_seed = []

    model_results[f'divcurves_{metric}'] = {}
    model_results[f'divcurves_pca_{metric}'] = {}
    model_results["divcurves_"+metric]['acc_mean'] = []
    model_results["divcurves_"+metric]['acc_std'] = []
    model_results[f'divcurves_pca_{metric}']['acc_mean'] = []
    model_results[f'divcurves_pca_{metric}']['acc_std'] = []

    for seed in range(n_seeds):
        np.random.seed(seed)
        random.seed(seed)

        # model_results[f'div_function_{metric}'] = {}

        print(f"Computing MAG with metric: {metric}")

        def dist_fn(G):
            return get_dist(G=G, X=None, X2=None, metric=metric, mode="structure")
        
        all_spreads, weights_lists = get_mag_graphs([g.copy() for g in G_nx], dist_fn, ts=[1], upsample_to=upsample_to, downsample_to=downsample_to, n_repeats=1, scores_method='random', compute_at_cardinality=compute_at_cardinality)

        mag_funs = get_funs_df(all_spreads)
        mag_funs_interpolated = mag_funs.interpolate(method='linear', axis='columns')

        mag_funs_per_seed.append(mag_funs_interpolated)

        # # compute clustering metrics based on euclidean distance of diversity curve vectors 
        # ari_kmeans_mean, ari_kmeans_std = compute_rand_kmeans(np.array(mag_funs_interpolated), y)
        # ari_agglo_mean, ari_agglo_std = compute_rand_agglo(np.array(mag_funs_interpolated), y, metric='euclidean')

        # means, stds = compute_clustering_scores_resample(np.array(mag_funs_interpolated), y)

        # model_results[f'divcurves_{metric}']['silh_mean'], model_results[f'divcurves_{metric}']['db_mean'], model_results[f'divcurves_{metric}']['ch_mean'] = means
        # model_results[f'divcurves_{metric}']['silh_std'], model_results[f'divcurves_{metric}']['db_std'], model_results[f'divcurves_{metric}']['ch_std'] = stds
        # model_results[f'divcurves_{metric}']['ari_agglo_mean'] = ari_agglo_mean
        # model_results[f'divcurves_{metric}']['ari_agglo_std'] = ari_agglo_std
        # model_results[f'divcurves_{metric}']['ari_kmeans_mean'] = ari_kmeans_mean
        # model_results[f'divcurves_{metric}']['ari_kmeans_std'] = ari_kmeans_std

        # computing accuracies
        if n_graphs > 1:
            test_accs = knn_classifier(np.array(mag_funs_interpolated), np.array(y), precomputed=False, groups=np.array(n_nodes))
        else:
            test_accs = knn_classifier(np.array(mag_funs_interpolated), np.array(y), precomputed=False)
        
        acc_per_seed["divcurves_"+metric].append(test_accs)
        model_results["divcurves_"+metric]['acc_mean'].append(np.mean(test_accs))
        model_results["divcurves_"+metric]['acc_std'].append(np.std(test_accs))

        # doing PCA with Euclidean distance between the diversity curves
        pca_embedding = PCA(n_components=2)
        X_pca = pca_embedding.fit_transform(np.array(mag_funs_interpolated))
        
        # computing clustering scores and accuracies on transformed data from PCA
        # all_clustering_results_euclidean(f'divcurves_pca_{metric}', model_results, X_pca, y)
        if n_graphs > 1:
            test_accs = knn_classifier(X_pca, np.array(y), precomputed=False, groups=np.array(n_nodes))
        else:
            test_accs = knn_classifier(X_pca, np.array(y), precomputed=False)
        
        acc_per_seed[f'divcurves_pca_{metric}'].append(test_accs)
        model_results[f'divcurves_pca_{metric}']['acc_mean'].append(np.mean(test_accs))
        model_results[f'divcurves_pca_{metric}']['acc_std'].append(np.std(test_accs))

        # #creating PCA plots
        # fig = sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette='tab10')
        # handles, labels = fig.get_legend_handles_labels()
        # fig.legend(
        #     handles=handles,
        #     labels= [hue_names[old_label] for old_label in labels],
        #     frameon=False,
        #     loc = (1.04, 0),
        #     bbox_to_anchor=(1, 0.5),
        # )
        # sns.despine()
        # plt.savefig(results_dir + f"{model}/divcurves_pca_{metric}.pdf", dpi=300, format='pdf', bbox_inches="tight")
        # plt.savefig(results_dir + f"{model}/divcurves_pca_{metric}.png", dpi=300, format='png', bbox_inches="tight")
        # plt.close()



###########################
### computing baselines ###
###########################


# storing results of model
res_df = pd.DataFrame.from_dict(model_results, orient='index')
all_results[model] = res_df
all_configs[model] = str(config)



Computing MAG with metric: shortest_path_distance
Fold 0: test accuracy = 1.0000 (k=5)
Fold 1: test accuracy = 1.0000 (k=5)
Fold 2: test accuracy = 1.0000 (k=5)
Fold 3: test accuracy = 1.0000 (k=5)
Fold 4: test accuracy = 1.0000 (k=5)
Fold 5: test accuracy = 1.0000 (k=5)
Fold 6: test accuracy = 1.0000 (k=5)
Fold 7: test accuracy = 1.0000 (k=5)
Fold 8: test accuracy = 1.0000 (k=5)
Fold 9: test accuracy = 1.0000 (k=5)

Mean test accuracy: 1.0000
Std test accuracy:  0.0000
Fold 0: test accuracy = 1.0000 (k=5)
Fold 1: test accuracy = 1.0000 (k=5)
Fold 2: test accuracy = 1.0000 (k=5)
Fold 3: test accuracy = 1.0000 (k=5)
Fold 4: test accuracy = 1.0000 (k=5)
Fold 5: test accuracy = 1.0000 (k=5)
Fold 6: test accuracy = 1.0000 (k=5)
Fold 7: test accuracy = 1.0000 (k=5)
Fold 8: test accuracy = 1.0000 (k=5)
Fold 9: test accuracy = 1.0000 (k=5)

Mean test accuracy: 1.0000
Std test accuracy:  0.0000
Computing MAG with metric: shortest_path_distance
Fold 0: test accuracy = 1.0000 (k=5)
Fold 1: test 

In [14]:
res_df

,acc_mean,acc_std
divcurves_shortest_path_distance,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
divcurves_pca_shortest_path_distance,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [12]:
mag_funs_per_seed[0]

index,1,2,3,4,5,6,7,8,9,10,...,20,21,22,23,24,25,26,27,28,29
graph,,,,,,,,,,,,,,,,,,,,,
0,1.0,2.000000,3.000000,4.000000,4.462117,4.924234,5.386351,5.830833,6.275315,6.719796,...,8.629344,8.634330,8.639317,8.744030,9.010264,9.121092,9.246541,9.351255,9.462083,9.592101
1,1.0,1.462117,1.906599,2.306254,2.753541,3.202895,3.663693,4.045132,4.436837,4.914690,...,5.853687,5.982184,6.110173,6.264238,6.392124,6.314489,6.302522,6.226299,6.220328,6.333851
2,1.0,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,7.462117,7.906599,8.368716,...,11.601129,11.867362,11.992811,12.114392,12.244410,12.510644,12.600720,12.866953,13.133187,13.399420
3,1.0,2.000000,3.000000,4.000000,5.000000,5.462117,5.728351,6.190468,6.652585,6.943861,...,9.404448,9.433942,9.639141,9.760721,9.882302,10.027185,10.125706,10.391939,10.482015,10.748249
4,1.0,2.000000,2.462117,2.906599,3.356722,3.812404,4.274521,4.733515,4.954693,5.419400,...,7.041499,7.075340,7.164211,7.316904,7.339078,7.347786,7.469367,7.628929,7.637467,7.639267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.604087,2.619369,2.639882,2.650451,2.665387,2.672741,2.684847,2.719264,2.730570,2.732695
176,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.588361,2.597333,2.611916,2.630935,2.665361,2.678200,2.694064,2.712920,2.729801,2.731711
177,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.564435,2.589879,2.598465,2.637566,2.641852,2.683659,2.715421,2.732767,2.747858,2.760739


In [11]:
mag_funs_per_seed[5]

index,1,2,3,4,5,6,7,8,9,10,...,20,21,22,23,24,25,26,27,28,29
graph,,,,,,,,,,,,,,,,,,,,,
0,1.0,2.000000,3.000000,4.000000,4.462117,4.924234,5.386351,5.830833,6.275315,6.719796,...,8.629344,8.895578,8.900564,9.026013,9.010264,9.135713,9.240426,9.224677,9.335506,9.446334
1,1.0,1.462117,1.906599,2.356722,2.753541,3.202895,3.593771,4.066258,4.459951,4.914690,...,5.853687,5.777367,5.884967,5.906876,5.845379,5.819506,5.953726,6.081459,6.218228,6.138845
2,1.0,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,7.462117,7.924234,8.368716,...,11.601129,11.722710,11.988943,12.114392,12.380626,12.646859,12.631110,12.721186,12.832015,13.098248
3,1.0,2.000000,3.000000,4.000000,5.000000,5.462117,5.906599,6.356722,6.818839,7.280956,...,9.292907,9.466024,9.639141,9.641932,9.627766,9.749347,9.836640,9.824093,10.090327,10.180403
4,1.0,2.000000,2.462117,2.906599,3.356722,3.812404,4.033559,4.497136,4.959253,5.423019,...,6.851619,7.129642,7.164211,7.319892,7.298120,7.419701,7.426027,7.516103,7.555851,7.520760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.573121,2.598840,2.607217,2.620905,2.643868,2.669062,2.680403,2.704736,2.730570,2.728440
176,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.580371,2.612058,2.632607,2.643481,2.670963,2.689377,2.699242,2.708012,2.729801,2.740468
177,1.0,1.462117,1.728351,1.901468,2.023048,2.113125,2.182537,2.237665,2.282505,2.319693,...,2.573121,2.583231,2.612686,2.650265,2.677505,2.683525,2.710233,2.727064,2.751953,2.760739


In [7]:
# mean and std (over all graphs) for l2 distance in difference seeds
avg_differences = []
std_differneces = []
for seed_i in range(n_seeds):
    for seed_j in range(seed_i+1, n_seeds):
        # getting a difference between the curves from random seed
        l2_differences = []
        for graph_index in mag_funs_per_seed[0].index:
            divcurvei, divcurvej = mag_funs_per_seed[seed_i].loc[graph_index], mag_funs_per_seed[seed_j].loc[graph_index]
            l2_differences.append((np.linalg.norm([divcurvei - divcurvej], ord=None)))
        avg_differences.append(np.mean(l2_differences))
        std_differneces.append(np.std(l2_differences))

In [8]:
avg_differences

[0.5688458555886619,
 0.5855756876381257,
 0.557764780007495,
 0.5556695467271942,
 0.545803296605282,
 0.5447224510983715,
 0.5599596068325422,
 0.5228411206311321,
 0.58240170872286,
 0.5804308289085482,
 0.5581080675730116,
 0.5369138888534101,
 0.5563989577980014,
 0.5463997670567612,
 0.5375488873185221,
 0.5551318395424983,
 0.5789463124843082,
 0.5614224763005726,
 0.5574120838000118,
 0.569037426734341,
 0.5698503827904221,
 0.5638883872108843,
 0.5670434640999508,
 0.5639442164222968,
 0.560607815817041,
 0.564203717352652,
 0.5315398055891124,
 0.5161205595069644,
 0.5433645284124137,
 0.5446811949529267,
 0.5453449677452414,
 0.5331930521067411,
 0.5219067536401412,
 0.5288807602148137,
 0.519905359604564,
 0.537441172544111,
 0.5455094012962878,
 0.5427457953535075,
 0.5514655682379765,
 0.5416009714446676,
 0.5207836193355728,
 0.5482678450571241,
 0.5596718853426295,
 0.5416756788471068,
 0.5398356131737649]

In [9]:
std_differneces

[0.5560364374379652,
 0.6040888361005037,
 0.5633689642714583,
 0.5483585892558095,
 0.55176187870338,
 0.5376485526330611,
 0.5729633211804187,
 0.48932114488632744,
 0.6178473846440069,
 0.5829031434023456,
 0.5720005267540426,
 0.5237878118506685,
 0.5416580406359209,
 0.5456649358409927,
 0.5587083418095127,
 0.5328967009812213,
 0.6070682683085599,
 0.6053920216463515,
 0.5885052022296104,
 0.5849613976356564,
 0.5919174393276254,
 0.5735664231085118,
 0.6035639560045138,
 0.608803693616139,
 0.5668602142130086,
 0.5877012083943342,
 0.5290870339932526,
 0.5140092683459166,
 0.5560051968439323,
 0.5423971228979659,
 0.5393581612380128,
 0.5174489248094528,
 0.50399424282657,
 0.49273473745707946,
 0.5300974274444132,
 0.5341814483820336,
 0.5444295090877181,
 0.550848242354708,
 0.5694917492808877,
 0.5405492456961973,
 0.5026518702627577,
 0.5417415423710912,
 0.5725977911594944,
 0.5359081782686137,
 0.5570097343321071]

# Results

## stability one curve 